# 05 · Instance Separation & Post-processing

**Goals**
- Turn the baseline's probability maps into better filament instances, without retraining
- Each idea targets an error type measured in notebook 04 and is kept only if it raises **PQ**
- Check the gain honestly on folds the settings were not tuned on, and measure the extra inference time

**Starting point (notebook 04, first run).** Instances = connected components of `p > threshold` on the disk,
components smaller than `min_area` dropped. Of the ground-truth filaments, 61% were matched, 19% **missed**,
10% **boundary** (one prediction, IoU ≤ 0.5), 5% **split** into several pieces and 4.5% **merged** with a
neighbour; PQ was still rising at the largest `min_area` tried, i.e. small spurious blobs cost a lot.

| Idea | Targets | How |
|---|---|---|
| Hysteresis threshold | split, boundary, spurious | keep a component of `p > low` only if it contains a confident pixel `p > high`: faint filament ends stay attached, faint blobs disappear |
| Gap grouping | split | pieces closer than `gap` px become one instance (the pixels themselves are not changed) |
| Boundary growth | boundary | grow every instance by `grow` px into the background, in case predictions are thinner than the annotated polygons |
| Test-time augmentation | all | average the probability over the 8 rotations/flips of the image (the disk is rotation invariant) |

**Inputs, all from previous notebooks:** the CSVs (01–02), the preprocessed images `cache/full_1024/` (02) and the
trained models + `baseline_runs.csv` (04). On Kaggle, attach the preprocessed-images dataset and the **output of
notebook 04** (*Add Input → Notebook Output Files*). `COS711_SMOKE=1` checks the pipeline on a few images on CPU.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch
except ImportError:
    %pip install -q segmentation-models-pytorch

In [ ]:
import json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from src.data import load_annotations
from src.preprocess import load_preprocessed, disk_mask
from src.augment import NO_AUG
from src.dataset import FilamentDataset
from src.model import build_model
from src.train import make_loader, predict, summarise
from src.postprocess import components, drop_small, upsample
from src.metrics import match_instances, pq_from_counts, error_types

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

S = 1024              # working resolution (as notebook 04)
FULL = 2048           # evaluation resolution (as notebook 04)
LIMB_CUT = 0.985      # pixels beyond 0.985 R are never filament
VAL_FOLD = 0

if SMOKE:
    OUT_DIR = ROOT / "outputs" / "smoke"
    BATCH = 2
    WORKERS = 0
else:
    OUT_DIR = ROOT
    BATCH = 8
    WORKERS = min(4, os.cpu_count())

FIG_DIR = OUT_DIR / "figures" / "05"
ART_OUT = OUT_DIR / "artifacts"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ART_OUT.mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)


def previous_output(rel):
    """A file written by an earlier notebook: an attached notebook output on Kaggle if there is one, else the repo copy."""
    candidates = []
    if IN_KAGGLE:
        candidates = sorted(Path("/kaggle/input").glob(f"**/{rel}"))
    candidates.append(OUT_DIR / rel)
    candidates.append(ROOT / rel)
    for path in candidates:
        if path.exists():
            print(rel, "←", path)
            return path
    message = f"{rel} not found: run the earlier notebooks first"
    if IN_KAGGLE:
        message += " and attach their outputs (Add Input → Notebook Output Files)"
    raise FileNotFoundError(message)


def preprocessed_dir(probe):
    """Folder that holds full_<S>/ with the preprocessed .npy files of notebook 02.
    Locally that is the repo's cache/; on Kaggle, wherever an attached input holds the files."""
    if IN_KAGGLE:
        candidates = []
        for f in sorted(Path("/kaggle/input").glob(f"**/{probe}")):
            candidates.append(f.parent)
    else:
        candidates = [ROOT / "cache" / f"full_{S}"]

    found = None
    for folder in candidates:
        if (folder / probe).exists():
            found = folder
            break
    if found is None:
        message = f"preprocessed images ({probe}) not found: run notebook 02"
        if IN_KAGGLE:
            message += f", upload cache/full_{S} as a Kaggle dataset and attach it"
        raise FileNotFoundError(message)
    print(f"cache/full_{S}", "←", found)

    # load_preprocessed reads <cache_dir>/full_<S>/<obs>.npy, so a folder with another name is linked under /tmp
    if found.name != f"full_{S}":
        link = Path("/tmp/cos711_cache") / f"full_{S}"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found, target_is_directory=True)
        found = link
    return found.parent


IMAGE_STATS_CSV = previous_output("artifacts/image_stats.csv")            # notebook 01
EXCLUDED_CSV = previous_output("artifacts/excluded_annotations.csv")      # notebook 02
FOLDS_CSV = previous_output("artifacts/folds.csv")                        # notebook 02
first_obs = pd.read_csv(FOLDS_CSV).obs.iloc[0]
CACHE_DIR = preprocessed_dir(f"{first_obs}.npy")                         # notebook 02
RUNS_CSV = previous_output("artifacts/baseline_runs.csv")                 # notebook 04

# Best fold-0 model of notebook 04 (not one of the cross-validation runs) and its tuned post-processing
runs = pd.read_csv(RUNS_CSV).drop_duplicates("name", keep="last")
fold0_runs = runs[(runs.fold == VAL_FOLD) & ~runs.name.str.contains("_fold")]
best = fold0_runs.sort_values("pq").iloc[-1]
BEST = best["name"]
T = float(best.threshold)
A = int(best.min_area)
CKPT = previous_output(f"checkpoints/{BEST}.pt")
print()
print(f"baseline: {BEST}  threshold {T}  min_area {A}  PQ {round(best.pq, 4)}")
if DEVICE.type == "cuda":
    print("device:", DEVICE, torch.cuda.get_device_name(), "| smoke:", SMOKE)
else:
    print("device:", DEVICE, "| smoke:", SMOKE)

## 1. Validation predictions

The same data, normalisation and validation fold as notebook 04 (the normalisation is recomputed with the
same code and sample, so it is identical). Probabilities are predicted once, plain and with test-time
augmentation, and every post-processing variant below is scored on them.

In [ ]:
images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(EXCLUDED_CSV)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(FOLDS_CSV)
geo_full = pd.read_csv(IMAGE_STATS_CSV).set_index("obs")[["cx", "cy", "radius"]]
paths = images.drop_duplicates("obs").set_index("obs").path

if SMOKE:   # a few observations per fold
    folds = folds.groupby("fold").head(3).reset_index(drop=True)


def split(fold):
    """(train observations, validation observations) when `fold` is the validation fold."""
    train = folds.loc[folds.fold != fold, "obs"].tolist()
    val = folds.loc[folds.fold == fold, "obs"].tolist()
    return train, val


# Input normalisation exactly as in notebook 04 (same images, same code, so the same values)
train_obs, val_obs = split(VAL_FOLD)
probe = FilamentDataset(train_obs, images, anns, geo_full, size=S, cache_dir=CACHE_DIR)
step = max(1, len(probe.obs) // 100)
values = []
for o in probe.obs[::step]:
    cx, cy, r = geo_full.loc[o]
    g = (cx * S / FULL, cy * S / FULL, r * S / FULL)
    x = load_preprocessed(paths[o], (cx, cy, r), S, cache_dir=CACHE_DIR)
    on_disk_pixels = x[disk_mask(x.shape, g[0], g[1], g[2], LIMB_CUT)]
    values.append(on_disk_pixels[::7])
values = np.concatenate(values)
med = float(np.median(values))
scale = float(1.4826 * np.median(np.abs(values - med)))
NORM = (med, scale)
print("normalisation (centre, scale):", (round(med, 4), round(scale, 4)))


def val_set(fold):
    """Validation dataset and loader of one fold."""
    _, va_obs = split(fold)
    va = FilamentDataset(va_obs, images, anns, geo_full, size=S, norm=NORM, aug=NO_AUG, cache_dir=CACHE_DIR)
    loader = make_loader(va, BATCH, train=False, workers=WORKERS)
    return va, loader


def load_model(path):
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    return m.eval()


def predict_tta(model, loader):
    """{obs: probability averaged over the 8 rotations / flips of the image} (float16, working resolution)."""
    out = {}
    use_fp16 = DEVICE.type == "cuda"
    with torch.no_grad():
        for x, _, idx in tqdm(loader, desc="TTA", leave=False):
            x = x.to(DEVICE)
            total = 0
            for k in range(4):                 # rotate by 0, 90, 180, 270 degrees
                for flip in (False, True):     # with and without a mirror flip
                    view = torch.rot90(x, k, (2, 3))
                    if flip:
                        view = view.flip(3)
                    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_fp16):
                        p = torch.sigmoid(model(view).float())
                    # undo the flip and the rotation so every prediction lines up with the original image
                    if flip:
                        p = p.flip(3)
                    total = total + torch.rot90(p, -k, (2, 3))
            mean_prob = (total / 8)[:, 0].cpu().numpy()
            ids = idx.tolist()
            for i in range(len(ids)):
                out[loader.dataset.obs[ids[i]]] = mean_prob[i].astype(np.float16)
    return out


model = load_model(CKPT)
va, va_dl = val_set(VAL_FOLD)


def gt_fn(o):
    return va.masks(o, FULL)


disks = {}
for o in va.obs:
    disks[o] = va.geo[o]

t0 = time.time()
probs = predict(model, va_dl)
t_plain = time.time() - t0

t0 = time.time()
probs_tta = predict_tta(model, va_dl)
t_tta = time.time() - t0

plain_ms = int(round(1000 * t_plain / len(probs)))
tta_ms = int(round(1000 * t_tta / len(probs)))
print(f"{len(probs)} validation images | network: plain {plain_ms} ms/image, TTA {tta_ms} ms/image")

## 2. Post-processing pipeline and evaluation

`instances(p, …)` turns a full-resolution probability map into an instance map; with only `high` and
`min_area` set it is exactly notebook 04's post-processing. Every variant is scored with the notebook 04
protocol: PQ at 2048 px against every annotation set of every validation observation, counts pooled.

In [ ]:
def disk_kernel(r):
    """Round structuring element of radius r (px)."""
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def make_settings(high, low=None, min_area=0, gap=0, grow=0):
    """One post-processing setting, as a dictionary:
    high: threshold (with hysteresis, the threshold for the confident seed pixels)
    low: hysteresis threshold for how far a filament extends (None = no hysteresis)
    min_area: drop instances smaller than this (px)
    gap: pieces closer than this (px) form one instance
    grow: grow every instance by this many px into the background"""
    return {"high": high, "low": low, "min_area": min_area, "gap": gap, "grow": grow}


def instances(p, on_disk, settings):
    """Instance map (0 = background) from a full-resolution probability map p and a boolean disk mask."""
    high = settings["high"]
    low = settings["low"]

    if low is None:
        fg = (p > high) & on_disk
    else:
        fg = (p > low) & on_disk
    lab, _ = components(fg)

    if low is not None:
        # Hysteresis: keep a component of p > low only if it contains a pixel with p > high
        keep = np.zeros(lab.max() + 1, bool)
        keep[np.unique(lab[fg & (p > high)])] = True
        keep[0] = False   # label 0 is the background
        fg = keep[lab]

    if settings["gap"] > 0:
        # Group pieces whose dilations touch; the pixels themselves stay the predicted ones
        dilated = cv2.dilate(fg.astype(np.uint8), disk_kernel(settings["gap"])) > 0
        groups, _ = components(dilated)
        lab = np.where(fg, groups, 0)
    elif low is not None:
        lab, _ = components(fg)

    lab = drop_small(lab, np.bincount(lab.ravel()), settings["min_area"])

    if settings["grow"] > 0:
        # Grow each instance into background pixels only, so neighbours are never overwritten
        grown = cv2.dilate(lab.astype(np.uint16), disk_kernel(settings["grow"])).astype(lab.dtype)
        lab = np.where((lab == 0) & on_disk, grown, lab)
    return lab


def describe(settings):
    """Short readable name of a setting, e.g. 'high=0.7 low=0.3 min_area=400'."""
    words = []
    for key in settings:
        value = settings[key]
        if value is not None and value != 0:
            words.append(f"{key}={value}")
    if not words:
        return "none"
    return " ".join(words)


def evaluate_configs(probs, configs, gt_fn, disks):
    """Per-image TP / FP / FN / IoU sums for every named setting in configs ({name: settings})."""
    rows = []
    for o in tqdm(probs, desc=f"{len(configs)} configs", leave=False):
        gts = gt_fn(o)
        p = upsample(probs[o], FULL)
        cx, cy, r = disks[o]
        on_disk = disk_mask(p.shape, cx, cy, r, LIMB_CUT)
        for config_name in configs:
            pred = instances(p, on_disk, configs[config_name])
            c = np.zeros(4)
            for gt in gts:
                m = match_instances(gt, pred)
                c += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
            rows.append({"obs": o, "config": config_name, "tp": c[0], "fp": c[1], "fn": c[2], "iou_sum": c[3]})
    return pd.DataFrame(rows)


def score(probs, configs, gt_fn=gt_fn, disks=disks):
    """Pooled PQ / SQ / RQ per setting, best first, with the settings as extra columns."""
    counts = evaluate_configs(probs, configs, gt_fn, disks)
    res = summarise(counts, by=["config"])
    res = res.join(pd.DataFrame.from_dict(configs, orient="index"))
    return res.sort_values("pq", ascending=False)


baseline = make_settings(high=T, min_area=A)
base_res = score(probs, {"baseline": baseline})
print(f"baseline reproduced: PQ {round(base_res.pq.iloc[0], 4)} (notebook 04: {round(best.pq, 4)})")
base_res[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 3. Hysteresis thresholding

`high` decides *whether* there is a filament, `low` how far it extends. A single threshold has to do both:
high enough to suppress faint false blobs, which then cuts faint filament ends off or splits the filament.
`min_area` is retuned together, since hysteresis already removes many small blobs.

In [ ]:
HIGHS = (0.6, 0.7, 0.8, 0.9)
LOWS = (None, 0.3, 0.4, 0.5)
MIN_AREAS = (200, 400, 800, 1600)

hyst_configs = {}
for h in HIGHS:
    for l in LOWS:
        if l is not None and l >= h:
            continue   # the low threshold must be below the high one
        for a in MIN_AREAS:
            settings = make_settings(high=h, low=l, min_area=a)
            hyst_configs[describe(settings)] = settings
hyst = score(probs, hyst_configs)
best_hyst = hyst_configs[hyst.index[0]]
print(f"best: {hyst.index[0]}  PQ {round(hyst.pq.iloc[0], 4)}  (baseline {round(base_res.pq.iloc[0], 4)})")

# PQ for each (high, low) pair, at the best min_area for that pair
table = hyst.copy()
table["low"] = table["low"].fillna(0)   # 0 stands for "no hysteresis"
grid = table.groupby(["high", "low"]).pq.max().unstack("low")
grid = grid.rename(columns={0: "none"})

fig, ax = plt.subplots(figsize=(6.5, 3.5))
im = ax.imshow(grid.values, cmap="viridis", aspect="auto")
ax.set(xticks=range(grid.shape[1]), xticklabels=grid.columns, yticks=range(grid.shape[0]), yticklabels=grid.index,
       xlabel="low (hysteresis)", ylabel="high", title="Validation PQ (best min_area per cell)")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        value = grid.values[i, j]
        if not np.isnan(value):
            ax.text(j, i, str(round(value, 3)), ha="center", va="center", fontsize=7, color="w")
plt.colorbar(im)
plt.tight_layout()
savefig("hysteresis_grid")
plt.show()
hyst.head(10)[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 4. Gap grouping

Filaments are often broken into pieces along their length. Pieces whose `gap`-px dilations touch become one
instance; their pixels are not changed, so boundaries (SQ) are unaffected. The risk is merging genuinely
separate neighbours (only 2.9% have a neighbour within 10 px, notebook 01). `min_area` is retuned with it,
since grouped instances are larger.

In [ ]:
GAPS = (0, 3, 6, 10, 15, 25)
a0 = best_hyst["min_area"]
gap_configs = {}
for g in GAPS:
    for a in sorted({a0 // 2, a0, a0 * 2}):   # min_area is retuned: grouped instances are larger
        settings = best_hyst.copy()
        settings["gap"] = g
        settings["min_area"] = a
        gap_configs[describe(settings)] = settings
gaps = score(probs, gap_configs)
best_gap = gap_configs[gaps.index[0]]
print(f"best: {gaps.index[0]}  PQ {round(gaps.pq.iloc[0], 4)}")

# Best result for each gap
per_gap = gaps.sort_values("pq").groupby("gap").tail(1)
per_gap = per_gap.set_index("gap").sort_index()[["pq", "sq", "rq", "min_area"]]
fig, ax = plt.subplots(figsize=(6, 3))
per_gap[["pq", "sq", "rq"]].plot(ax=ax, marker="o")
ax.set(xlabel="gap (full-res px)", ylabel="validation score", title="Gap grouping (best min_area per gap)")
plt.tight_layout()
savefig("gap_grouping")
plt.show()
per_gap

## 5. Boundary growth

If predicted filaments are systematically thinner than the annotated polygons, boundary errors (one
prediction, IoU ≤ 0.5) can be turned into matches by growing every instance by a few pixels.

In [ ]:
GROWS = (0, 1, 2, 3, 5, 8)
grow_configs = {}
for g in GROWS:
    settings = best_gap.copy()
    settings["grow"] = g
    grow_configs[describe(settings)] = settings
grows = score(probs, grow_configs)
best_grow = grow_configs[grows.index[0]]
print(f"best: {grows.index[0]}  PQ {round(grows.pq.iloc[0], 4)}")
grows.sort_values("grow")[["grow", "pq", "sq", "rq", "tp", "fp", "fn"]]

## 6. Test-time augmentation

The model is applied to the 8 rotations / flips of each image and the probabilities are averaged. The disk is
rotation invariant, so every view is a valid image. Averaging changes the probability scale, so the thresholds
are retuned around the chosen pipeline.

In [ ]:
# Averaging over 8 views changes the probability scale, so the high threshold is retuned around the chosen one
tta_configs = {}
for dh in (-0.1, 0.0, 0.1):
    h = round(min(0.95, max(0.1, best_grow["high"] + dh)), 2)
    if best_grow["low"] is not None and best_grow["low"] >= h:
        continue
    settings = best_grow.copy()
    settings["high"] = h
    tta_configs[describe(settings)] = settings
tta = score(probs_tta, tta_configs)

plain_final = grows.pq.iloc[0]
use_tta = tta.pq.iloc[0] > plain_final
if use_tta:
    FINAL = tta_configs[tta.index[0]]
else:
    FINAL = best_grow
print(f"plain {round(plain_final, 4)} vs TTA {round(tta.pq.iloc[0], 4)} ({tta.index[0]}) → use TTA: {use_tta}")
tta[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 7. What changed: error types, baseline vs final

In [ ]:
if use_tta:
    final_probs = probs_tta
else:
    final_probs = probs


def error_breakdown(probs, settings):
    """Share (%) of ground-truth filaments matched / missed / merged / split / boundary, and the spurious count."""
    err = []
    for o in tqdm(probs, desc="error types", leave=False):
        p = upsample(probs[o], FULL)
        cx, cy, r = disks[o]
        pred = instances(p, disk_mask(p.shape, cx, cy, r, LIMB_CUT), settings)
        for gt in gt_fn(o):
            err.append(error_types(gt, pred))
    tot = pd.DataFrame(err).sum()
    fates = ["matched", "missed", "merged", "split", "boundary"]
    shares = 100 * tot[fates] / tot[fates].sum()
    shares["spurious (count)"] = tot.spurious
    return shares


errs = pd.DataFrame({"baseline": error_breakdown(probs, baseline), "final": error_breakdown(final_probs, FINAL)})
fig, ax = plt.subplots(figsize=(7, 3))
errs.drop("spurious (count)").plot.bar(ax=ax)
ax.set(ylabel="% of GT filaments", title="Fate of ground-truth filaments (validation)")
plt.xticks(rotation=0)
plt.tight_layout()
savefig("error_types_before_after")
plt.show()
errs.round(1)

## 8. Honest check on folds 1–4

Every setting above was chosen on fold 0, so the fold-0 gain is optimistic. Notebook 04's cross-validation
models (`<best>_fold1…4`) were never used for tuning: the baseline and the final pipeline are applied to each
with **fixed** settings, and the gain is reported as mean ± std over the folds. Skipped if notebook 04 was run
without cross-validation.

In [ ]:
cv_rows = []
for f in sorted(folds.fold.unique()):
    if f == VAL_FOLD:
        continue
    try:
        ckpt = previous_output(f"checkpoints/{BEST}_fold{f}.pt")
    except FileNotFoundError:
        print(f"fold {f}: no checkpoint, skipped")
        continue

    m = load_model(ckpt)
    va_f, dl_f = val_set(f)

    def gt_f(o):
        return va_f.masks(o, FULL)

    disks_f = {}
    for o in va_f.obs:
        disks_f[o] = va_f.geo[o]

    pr_plain = predict(m, dl_f)
    if use_tta:
        pr_final = predict_tta(m, dl_f)
    else:
        pr_final = pr_plain
    b = score(pr_plain, {"baseline": baseline}, gt_f, disks_f).pq.iloc[0]
    fi = score(pr_final, {"final": FINAL}, gt_f, disks_f).pq.iloc[0]
    cv_rows.append({"fold": f, "baseline": b, "final": fi, "gain": fi - b})

    del m
    torch.cuda.empty_cache()

if cv_rows:
    cv = pd.DataFrame(cv_rows).set_index("fold")
    fold_list = list(cv.index)
    base_text = f"{round(cv.baseline.mean(), 3)} ± {round(cv.baseline.std(), 3)}"
    final_text = f"{round(cv.final.mean(), 3)} ± {round(cv.final.std(), 3)}"
    gain_text = f"{round(cv.gain.mean(), 3)} ± {round(cv.gain.std(), 3)}"
    print(f"folds {fold_list}: baseline {base_text} → final {final_text}  (gain {gain_text})")
    display(cv)
else:
    cv = None

## 9. Cost: post-processing and inference time

Post-processing time per image at full resolution, and the network time with and without TTA (section 1).

In [ ]:
def postprocess_ms(probs, settings, n=20):
    """Average post-processing time per image (ms) over the first n images."""
    times = []
    for o in list(probs)[:n]:
        t0 = time.perf_counter()
        p = upsample(probs[o], FULL)
        cx, cy, r = disks[o]
        instances(p, disk_mask(p.shape, cx, cy, r, LIMB_CUT), settings)
        times.append(time.perf_counter() - t0)
    return 1000 * float(np.mean(times))


if use_tta:
    final_network_s = t_tta
else:
    final_network_s = t_plain
cost = pd.DataFrame({
    "network ms / image": [1000 * t_plain / len(probs), 1000 * final_network_s / len(probs)],
    "post-processing ms / image": [postprocess_ms(probs, baseline), postprocess_ms(final_probs, FINAL)],
}, index=["baseline", "final"])
cost["total ms / image"] = cost.sum(axis=1)
cost.round(1)

## Summary: Panoptic Quality

Each step is added on top of the previous one (fold 0; settings tuned there). The final settings are saved to
`artifacts/postprocess.json` for the inference notebook.

In [ ]:
steps = pd.DataFrame({
    "PQ (fold 0)": [base_res.pq.iloc[0], hyst.pq.iloc[0], gaps.pq.iloc[0], grows.pq.iloc[0], tta.pq.iloc[0]],
    "settings": [describe(baseline), hyst.index[0], gaps.index[0], grows.index[0], tta.index[0]],
}, index=["baseline (notebook 04)", "+ hysteresis", "+ gap grouping", "+ boundary growth", "+ TTA"])
display(steps)

final_name = describe(FINAL)
if use_tta:
    final_name = final_name + " + TTA"
final_pq = max(plain_final, tta.pq.iloc[0])
print(f"Final: {final_name}, PQ {round(final_pq, 4)} on fold 0 (baseline {round(base_res.pq.iloc[0], 4)})")
if cv is not None:
    print(f"Folds {list(cv.index)} (not tuned on): baseline {round(cv.baseline.mean(), 3)} ± {round(cv.baseline.std(), 3)}"
          f" → final {round(cv.final.mean(), 3)} ± {round(cv.final.std(), 3)}")

# Settings for the inference notebook
chosen = {"model": BEST, "tta": bool(use_tta)}
chosen.update(FINAL)
with open(ART_OUT / "postprocess.json", "w") as fh:
    json.dump(chosen, fh, indent=1)